# C6. The potential $\theta_\alpha$ vanishes on the leaves of $P_\alpha$

Companion notebook to the bachelor's thesis *El formalismo ODM en sistemas híbridos clásico-cuánticos* (Santiago Puyol Miano, Universidad de Zaragoza, 2026).

The quantization along the tilted polarization $P_\alpha$ is well posed because the symplectic potential used for it vanishes on the leaf directions of $P_\alpha$. The thesis takes $\theta_\alpha=(g_\alpha^{-1})^*\Theta$, the pullback of $\Theta=\lambda_x\,dx+\lambda_p\,dp$ along the inverse of $g_\alpha$. This notebook checks

1. that $\theta_\alpha-\Theta=du_\alpha$ with $u_\alpha=\tfrac12\sin\alpha\cos\alpha\,(p^2-\lambda_p^2)-\sin^2\alpha\;p\lambda_p$;
2. that $\theta_\alpha$ vanishes on both leaf directions of $P_\alpha$ for every $\alpha$, while the pullback $g_\alpha^*\Theta$ along $g_\alpha$ itself (the identity of C2) vanishes on them only at $\alpha=0$ and $\alpha=\pi/2$;
3. numerically, in a truncated Fock basis, the operator identity used to quantize along $P_\alpha$: conjugating $\hat p$ and $\hat\lambda_p$ by $\mathcal U(g_\alpha)$ gives operators $\hat p_\alpha,\hat w_\alpha$ with the rotated closed forms and $[\hat p_\alpha,\hat w_\alpha]=i$.

**Kernel:** Python 3 with sympy and numpy.

## Notation

- Coordinates $(x,p,\lambda_x,\lambda_p)$ on $\Xi=T^*\mathbb R^2$, in this order; $a$ is the angle $\alpha$.
- A 1-form is stored as its coefficients on $(dx,dp,d\lambda_x,d\lambda_p)$, so $\Theta=(\lambda_x,\lambda_p,0,0)$.
- $g_\alpha$ rotates the pair $(p,\lambda_p)$ by $\alpha$, and $g_\alpha^{-1}=g_{-\alpha}$.
- $P_\alpha=\operatorname{span}\{\partial_{\lambda_x},\,W_\alpha\}$ with $W_\alpha=\sin\alpha\,\partial_p+\cos\alpha\,\partial_{\lambda_p}$.

In [1]:
import sympy as sp
import numpy as np

a = sp.symbols('alpha', real=True)
x, p, lx, lp = sp.symbols('x p lambda_x lambda_p', real=True)

# g_alpha and its inverse g_{-alpha}, coordinate order (x, p, lx, lp)
g = sp.Matrix([
    [1, 0, 0, 0],
    [0, sp.cos(a), 0, sp.sin(a)],
    [0, 0, 1, 0],
    [0, -sp.sin(a), 0, sp.cos(a)],
])
ginv = sp.simplify(g.inv())
assert sp.simplify(ginv - g.subs(a, -a)) == sp.zeros(4, 4), \
    "g_alpha^{-1} != g_{-alpha}"

z = sp.Matrix([x, p, lx, lp])


def pullback_1form(matrix_map):
    """Pullback of Theta = lx dx + lp dp along the linear map `matrix_map`.
    Theta is evaluated at the image point and contracted with the Jacobian,
    which is the matrix itself because the map is linear. Returns the
    coefficients on (dx, dp, dlx, dlp)."""
    zc = matrix_map * z          # coordinates of the image point, as functions of z
    Theta_at_image = sp.Matrix([zc[2], zc[3], 0, 0])
    # (h^*Theta)(z) = Theta_{h(z)} . dh|_z, with dh = matrix_map
    return sp.simplify((Theta_at_image.T * matrix_map).T)

## 1. Both pullbacks of $\Theta$ differ from $\Theta$ by an exact form

- Along $g_\alpha$: $g_\alpha^*\Theta-\Theta=d\tilde u_\alpha$ with $\tilde u_\alpha=\tfrac12\sin\alpha\cos\alpha\,(\lambda_p^2-p^2)-\sin^2\alpha\,p\lambda_p$, as in C2.
- Along $g_\alpha^{-1}$: $\theta_\alpha-\Theta=du_\alpha$ with $u_\alpha=\tfrac12\sin\alpha\cos\alpha\,(p^2-\lambda_p^2)-\sin^2\alpha\,p\lambda_p$.

Both functions are $0$ at $\alpha=0$ and $-p\lambda_p$ at $\alpha=\pi/2$.

In [2]:
Theta = sp.Matrix([lx, lp, 0, 0])

# Pullback along g_alpha
theta_g = pullback_1form(g)
diff_g = sp.simplify(theta_g - Theta)
u_tilde = sp.Rational(1, 2) * sp.sin(a) * sp.cos(a) * (lp**2 - p**2) - sp.sin(a)**2 * p * lp
du_tilde = sp.Matrix([sp.diff(u_tilde, v) for v in (x, p, lx, lp)])
assert sp.simplify(diff_g - du_tilde) == sp.zeros(4, 1), \
    f"g_alpha^*Theta - Theta != d(u_tilde): {sp.simplify(diff_g - du_tilde)}"
print("OK: g_alpha^*Theta - Theta = d(u_tilde)")

# Pullback along g_alpha^{-1}: theta_alpha
theta_ginv = pullback_1form(ginv)
u_alpha = sp.Rational(1, 2) * sp.sin(a) * sp.cos(a) * (p**2 - lp**2) - sp.sin(a)**2 * p * lp
du_alpha = sp.Matrix([sp.diff(u_alpha, v) for v in (x, p, lx, lp)])
diff_ginv = sp.simplify(theta_ginv - Theta)
assert sp.simplify(diff_ginv - du_alpha) == sp.zeros(4, 1), \
    f"theta_alpha - Theta != d(u_alpha): {sp.simplify(diff_ginv - du_alpha)}"
print("OK: theta_alpha - Theta = (g_alpha^{-1})^*Theta - Theta = d(u_alpha)")

assert sp.simplify(u_alpha.subs(a, 0)) == 0, "u_alpha(0) != 0"
assert sp.simplify(u_alpha.subs(a, sp.pi / 2) - (-p * lp)) == 0, "u_alpha(pi/2) != -p*lp"
print("OK: u_0 = 0 and u_{pi/2} = -p*lambda_p")

OK: g_alpha^*Theta - Theta = d(u_tilde)
OK: theta_alpha - Theta = (g_alpha^{-1})^*Theta - Theta = d(u_alpha)
OK: u_0 = 0 and u_{pi/2} = -p*lambda_p


## 2. Which potential vanishes on the leaves of $P_\alpha$

A 1-form $c_x\,dx+c_p\,dp+c_{\lambda_x}d\lambda_x+c_{\lambda_p}d\lambda_p$ takes the value $c_{\lambda_x}$ on $\partial_{\lambda_x}$ and $\sin\alpha\,c_p+\cos\alpha\,c_{\lambda_p}$ on $W_\alpha$. Both potentials vanish on $\partial_{\lambda_x}$. On $W_\alpha$, $\theta_\alpha$ vanishes for every $\alpha$, and $g_\alpha^*\Theta$ vanishes only at $\alpha=0$ and $\alpha=\pi/2$, so a check made only at the endpoints cannot tell the two potentials apart.

In [3]:
def apply_to_leaf(theta_coeffs, alpha_val=None):
    c_x, c_p, c_lx, c_lp = theta_coeffs
    on_dlx = c_lx
    on_W = sp.sin(a) * c_p + sp.cos(a) * c_lp
    if alpha_val is not None:
        on_dlx = on_dlx.subs(a, alpha_val)
        on_W = on_W.subs(a, alpha_val)
    return sp.simplify(on_dlx), sp.simplify(on_W)


g_dlx, g_W = apply_to_leaf(theta_g)
ginv_dlx, ginv_W = apply_to_leaf(theta_ginv)

assert sp.simplify(g_dlx) == 0, "g_alpha^*Theta does not vanish on d/dlx"
assert sp.simplify(ginv_dlx) == 0, "theta_alpha does not vanish on d/dlx"
print("OK: both potentials vanish on d/dlambda_x")

g_W_simplified = sp.simplify(g_W)
ginv_W_simplified = sp.simplify(ginv_W)
assert sp.simplify(ginv_W_simplified) == 0, \
    f"theta_alpha does not vanish on W_alpha for general alpha: {ginv_W_simplified}"
print("OK: theta_alpha vanishes on W_alpha for every alpha")

assert sp.simplify(g_W_simplified) != 0, \
    "g_alpha^*Theta vanishes on W_alpha for general alpha"
for alpha_val in [0, sp.pi / 2]:
    assert sp.simplify(g_W_simplified.subs(a, alpha_val)) == 0, \
        f"g_alpha^*Theta should vanish on W_alpha at alpha={alpha_val}"
mid_val = sp.simplify(g_W_simplified.subs(a, sp.pi / 4))
assert mid_val != 0, "g_alpha^*Theta vanishes on W_alpha at alpha=pi/4"
print(f"OK: g_alpha^*Theta vanishes on W_alpha only at alpha = 0 and pi/2; "
      f"at alpha = pi/4 it equals {mid_val}")

OK: both potentials vanish on d/dlambda_x
OK: theta_alpha vanishes on W_alpha for every alpha
OK: g_alpha^*Theta vanishes on W_alpha only at alpha = 0 and pi/2; at alpha = pi/4 it equals sqrt(2)*(lambda_p - p)/2


## 3. Conjugation by $\mathcal U(g_\alpha)$ in the Fock basis (numerical)

As in C5, $\mathcal U_\alpha=e^{-i\alpha(N+1/2)}$ on a Fock basis of the $(p,\lambda_p)$ block with $[\hat p,\hat\lambda_p]=i$, now truncated at dimension 80. The cell checks that $\hat p_\alpha=\mathcal U\hat p\,\mathcal U^{-1}$ and $\hat w_\alpha=\mathcal U\hat\lambda_p\,\mathcal U^{-1}$ equal $\cos\alpha\,\hat p-\sin\alpha\,\hat\lambda_p$ and $\sin\alpha\,\hat p+\cos\alpha\,\hat\lambda_p$, and that $[\hat p_\alpha,\hat w_\alpha]=i$, at $\alpha=0.2,\ \pi/6,\ 1,\ \pi/3,\ 1.4$. These formulas enter the proof that transporting the vertical quantization with $\mathcal U(g_\alpha)$ agrees with geometric quantization along $P_\alpha$.

In [4]:
dim = 80
n_arr = np.arange(dim)
ad = np.diag(np.sqrt(np.arange(1, dim)), -1)   # a^dagger
aa = ad.conj().T                                # a
P_op = (aa + ad) / np.sqrt(2)
Lp_op = -1j * (aa - ad) / np.sqrt(2)
comm_P_Lp = P_op @ Lp_op - Lp_op @ P_op
interior = slice(0, dim - 2)
assert np.allclose(comm_P_Lp[interior, interior], 1j * np.eye(dim)[interior, interior], atol=1e-8), \
    "[P,Lp] != i in Fock truncation"

N = np.diag(n_arr)


def U(alpha):
    G = N + 0.5 * np.eye(dim)
    # exp(-i alpha G), diagonal in the Fock basis
    return np.diag(np.exp(-1j * alpha * np.diag(G)))


for alpha_val in [0.2, np.pi / 6, 1.0, np.pi / 3, 1.4]:
    Ua = U(alpha_val)
    Uainv = U(-alpha_val)
    p_alpha = Ua @ P_op @ Uainv
    w_alpha = Ua @ Lp_op @ Uainv

    # closed forms used in the proof
    p_alpha_expected = np.cos(alpha_val) * P_op - np.sin(alpha_val) * Lp_op
    w_alpha_expected = np.sin(alpha_val) * P_op + np.cos(alpha_val) * Lp_op

    assert np.allclose(p_alpha[interior, interior], p_alpha_expected[interior, interior], atol=1e-6), \
        f"p_alpha mismatch at alpha={alpha_val}"
    assert np.allclose(w_alpha[interior, interior], w_alpha_expected[interior, interior], atol=1e-6), \
        f"w_alpha mismatch at alpha={alpha_val}"

    comm_pw = p_alpha @ w_alpha - w_alpha @ p_alpha
    assert np.allclose(comm_pw[interior, interior], 1j * np.eye(dim)[interior, interior], atol=1e-6), \
        f"[p_alpha, w_alpha] != i at alpha={alpha_val}: max err " \
        f"{np.max(np.abs(comm_pw[interior, interior] - 1j*np.eye(dim)[interior, interior]))}"

print("OK: p_alpha = U p U^-1 and w_alpha = U lp U^-1 match the rotated closed forms "
      "and satisfy [p_alpha, w_alpha] = i at every tested alpha")

OK: p_alpha = U p U^-1 and w_alpha = U lp U^-1 match the rotated closed forms and satisfy [p_alpha, w_alpha] = i at every tested alpha


In [5]:
# Every assert above has passed if this cell runs.
print('C6: all checks passed.')

C6: all checks passed.
